# Match Reviewed Labels to History Data

This notebook matches the reviewed pushback labels generated from the raw surveillance data to the corresponding OpenSky history trajectories.

The workflow:

1. Load and preprocess the history data.
2. Load the reviewed raw-data label mapping.
3. Match raw flight IDs to history flight IDs using callsign and timestamps.
4. Trim history trajectories that begin before the corresponding raw trajectory.
5. Transfer the pushback time intervals to the history trajectories as pointwise labels.
6. Store the matched dataset for the next processing step.


## 1. Imports

In [31]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from traffic.core import Traffic

warnings.filterwarnings("ignore", category=RuntimeWarning)


## 2. Configuration

In [32]:
airport_str = "LSZH"
season = "Spring"  # Spring, Summer, Autumn, or Winter

max_workers = 10

valid_seasons = {"Spring", "Summer", "Autumn", "Winter"}

if season not in valid_seasons:
    raise ValueError(
        f"Invalid season '{season}'. Expected one of: {sorted(valid_seasons)}"
    )

PROJECT_ROOT = Path.home() / "Projects" / "OSN_pushback_detection"
DATA_ROOT = PROJECT_ROOT / "data"

history_dir = (
    DATA_ROOT
    / "train_validation"
    / "01_source"
    / "history"
    / airport_str
    / season
)

reviewed_dir = (
    DATA_ROOT
    / "train_validation"
    / "03_reviewed"
    / airport_str
    / season
)

matched_dir = (
    DATA_ROOT
    / "train_validation"
    / "04_matched"
    / airport_str
    / season
)
matched_dir.mkdir(parents=True, exist_ok=True)

mapping_path = (
    reviewed_dir
    / f"{airport_str}_{season}_mapping_set.csv"
)

matched_path = (
    matched_dir
    / f"{airport_str}_{season}_matched.parquet"
)

print(f"Airport:        {airport_str}")
print(f"Season:         {season}")
print(f"History data:   {history_dir}")
print(f"Reviewed labels:{mapping_path}")
print(f"Matched output: {matched_path}")


Airport:        LSZH
Season:         Spring
History data:   /cluster/home/hanv/Projects/OSN_pushback_detection/data/train_validation/01_source/history/LSZH/Spring
Reviewed labels:/cluster/home/hanv/Projects/OSN_pushback_detection/data/train_validation/03_reviewed/LSZH/Spring/LSZH_Spring_mapping_set.csv
Matched output: /cluster/home/hanv/Projects/OSN_pushback_detection/data/train_validation/04_matched/LSZH/Spring/LSZH_Spring_matched.parquet


## 3. Load History Data

Load all history parquet files for the selected airport and season.

In [33]:
files = sorted(history_dir.glob("*.parquet"))

if not files:
    raise FileNotFoundError(
        f"No history parquet files found in {history_dir}"
    )

print(f"Found {len(files)} history files")

df = pd.concat(
    (pd.read_parquet(file) for file in files),
    ignore_index=True,
)

trajs = Traffic(df)

print(f"Loaded {len(trajs.data):,} history points")


Found 1 history files
Loaded 7,747,787 history points


## 4. Normalize Data Types and Resample

Normalize the history data types, assign deterministic flight IDs, and resample the trajectories to 1 Hz.

`assign_id()` intentionally uses a single worker to keep flight ID assignment deterministic.

In [34]:
def convert_timestamp(value):
    if isinstance(value, (float, int)) and value > 1e9:
        return pd.to_datetime(value, unit="s", errors="coerce")

    if isinstance(value, pd.Timestamp):
        return value

    return pd.NaT


In [35]:
trajs.data["timestamp"] = trajs.data["timestamp"].apply(convert_timestamp)
trajs.data["hour"] = trajs.data["hour"].apply(convert_timestamp)

trajs.data["onground"] = trajs.data["onground"].astype(np.bool_)
trajs.data["alert"] = trajs.data["alert"].astype(np.bool_)
trajs.data["spi"] = trajs.data["spi"].astype(np.bool_)

trajs.data["last_position"] = trajs.data["last_position"].astype(np.float64)

trajs.data["latitude"] = trajs.data["latitude"].astype("double")
trajs.data["longitude"] = trajs.data["longitude"].astype("double")
trajs.data["groundspeed"] = trajs.data["groundspeed"].astype("double")
trajs.data["track"] = trajs.data["track"].astype("double")
trajs.data["vertical_rate"] = trajs.data["vertical_rate"].astype("double")
trajs.data["altitude"] = trajs.data["altitude"].astype("double")
trajs.data["geoaltitude"] = trajs.data["geoaltitude"].astype("double")
trajs.data["lastcontact"] = trajs.data["lastcontact"].astype("double")

trajs.data["icao24"] = trajs.data["icao24"].astype("object")
trajs.data["callsign"] = trajs.data["callsign"].astype("object")
trajs.data["squawk"] = trajs.data["squawk"].astype("object")


In [36]:
trajs = trajs.assign_id().eval(
    max_workers=1,
    desc="Assigning history flight IDs",
)

trajs = trajs.resample("1s").eval(
    max_workers=max_workers,
    desc="Resampling history trajectories",
)

print(f"Preprocessed {trajs.data['flight_id'].nunique():,} history flights")


Assigning history flight IDs:   0%|          | 0/6245 [00:00<?, ?it/s]

Resampling history trajectories:   0%|          | 0/6245 [00:00<?, ?it/s]

Preprocessed 6,245 history flights


## 5. Load Reviewed Label Mapping

Load the final reviewed mapping generated by `02_label_raw_data`.

In [37]:
if not mapping_path.exists():
    raise FileNotFoundError(
        f"Reviewed mapping file not found: {mapping_path}"
    )

mapping_set = pd.read_csv(mapping_path)

datetime_columns = [
    "Start",
    "Stop",
    "Pushback Start",
    "Pushback Stop",
]

for column in datetime_columns:
    mapping_set[column] = pd.to_datetime(
        mapping_set[column],
        utc=True,
        errors="coerce",
    )

print(f"Loaded {len(mapping_set):,} reviewed flights")

mapping_set.head()


Loaded 2,504 reviewed flights


,Unnamed: 0,Flight ID Raw,Callsign,Start,Stop,Pushback,Pushback Start,Pushback Stop
0,0,9AJIP_5344,9AJIP,2026-04-20 11:38:35+00:00,2026-04-20 11:56:41+00:00,False,NaT,NaT
1,1,AAL93_6131,AAL93,2026-04-26 09:37:52+00:00,2026-04-26 09:52:31+00:00,True,2026-04-26 09:41:05+00:00,2026-04-26 09:42:17+00:00
2,2,AAL93_6129,AAL93,2026-04-24 09:42:28+00:00,2026-04-24 09:58:50+00:00,True,2026-04-24 09:46:24+00:00,2026-04-24 09:47:55+00:00
3,3,AAL93_6135,AAL93,2026-04-22 09:44:44+00:00,2026-04-22 10:05:42+00:00,True,2026-04-22 09:44:57+00:00,2026-04-22 09:46:16+00:00
4,4,ACA881_6148,ACA881,2026-04-26 11:15:04+00:00,2026-04-26 11:37:26+00:00,True,2026-04-26 11:18:31+00:00,2026-04-26 11:19:44+00:00


## 6. Match Raw Flight IDs to History Flight IDs

History flights are matched to reviewed raw-data flights by:

- identical callsign;
- raw stop time within ±2 minutes of the history stop time;
- smallest absolute start-time difference if multiple candidates remain.

In [38]:
def map_ids(flight):
    callsign = flight.callsign

    start = flight.stop - pd.Timedelta("2 min")
    stop = flight.stop + pd.Timedelta("2 min")

    match = mapping_set[
        (mapping_set["Callsign"] == callsign)
        & (mapping_set["Stop"] >= start)
        & (mapping_set["Stop"] <= stop)
    ]

    if len(match) == 0:
        return None

    if len(match) > 1:
        match = match.iloc[
            (match["Start"] - flight.start)
            .abs()
            .argsort()
        ]

    return {
        "Flight ID Raw": match.iloc[0]["Flight ID Raw"],
        "Flight ID New": flight.flight_id,
        "New Start": flight.start,
        "New Stop": flight.stop,
    }


mapping = (
    trajs.iterate_lazy()
    .pipe(map_ids)
    .eval(
        max_workers=max_workers,
        desc="Matching history flights",
    )
)

mapping


Matching history flights:   0%|          | 0/6245 [00:00<?, ?it/s]

,Flight ID Raw,Flight ID New,New Start,New Stop
0,9AJIP_5344,9AJIP_5415,2026-04-20 11:38:47+00:00,2026-04-20 11:56:41+00:00
1,AAL93_6127,AAL93_6214,2026-04-20 09:41:35+00:00,2026-04-20 10:10:07+00:00
2,AAL93_6129,AAL93_6216,2026-04-24 09:42:47+00:00,2026-04-24 09:58:50+00:00
3,AAL93_6131,AAL93_6218,2026-04-26 09:37:54+00:00,2026-04-26 09:52:32+00:00
4,AAL93_6133,AAL93_6220,2026-04-21 09:45:43+00:00,2026-04-21 10:10:17+00:00
...,...,...,...,...
2485,VLG61XZ_157,VLG61XZ_158,2026-04-26 19:24:46+00:00,2026-04-26 19:38:32+00:00
2486,VLG61XZ_194,VLG61XZ_196,2026-04-25 19:34:35+00:00,2026-04-25 19:51:54+00:00
2487,VLG61XZ_204,VLG61XZ_207,2026-04-23 19:40:08+00:00,2026-04-23 19:53:56+00:00
2488,VLJ513B_279,VLJ513B_276,2026-04-26 09:30:00+00:00,2026-04-26 09:43:24+00:00


## 7. Merge Flight-ID Mapping

Merge the matched history IDs back into the reviewed label table and calculate the difference between the raw-data start time and the history-data start time.

In [39]:
mapping_set = mapping_set.merge(
    mapping,
    on="Flight ID Raw",
    how="left",
)

mapping_set["New Start"] = pd.to_datetime(
    mapping_set["New Start"],
    utc=True,
    errors="coerce",
)
mapping_set["New Stop"] = pd.to_datetime(
    mapping_set["New Stop"],
    utc=True,
    errors="coerce",
)

mapping_set["Time Difference"] = (
    mapping_set["Start"] - mapping_set["New Start"]
).dt.total_seconds()

matched_count = mapping_set["Flight ID New"].notna().sum()
unmatched_count = mapping_set["Flight ID New"].isna().sum()

print(f"Matched reviewed flights:   {matched_count:,}")
print(f"Unmatched reviewed flights: {unmatched_count:,}")

mapping_set.sort_values(
    "Time Difference",
    ascending=False,
).head(10)


Matched reviewed flights:   2,490
Unmatched reviewed flights: 14


,Unnamed: 0,Flight ID Raw,Callsign,Start,Stop,Pushback,Pushback Start,Pushback Stop,Flight ID New,New Start,New Stop,Time Difference
914,914,MSR774_010,MSR774,2026-04-20 14:28:22+00:00,2026-04-20 14:43:55+00:00,True,2026-04-20 14:28:22+00:00,2026-04-20 14:29:10+00:00,MSR774_008,2026-04-20 12:34:50+00:00,2026-04-20 14:43:55+00:00,6812.0
1791,1791,SWR4ZW_3281,SWR4ZW,2026-04-23 11:00:52+00:00,2026-04-23 11:18:03+00:00,False,NaT,NaT,SWR4ZW_3320,2026-04-23 09:13:58+00:00,2026-04-23 11:18:04+00:00,6414.0
2292,2292,SWR9GE_3302,SWR9GE,2026-04-21 11:53:36+00:00,2026-04-21 12:06:10+00:00,False,NaT,NaT,SWR9GE_3340,2026-04-21 10:11:34+00:00,2026-04-21 12:06:11+00:00,6122.0
1878,1878,SWR5BW_2832,SWR5BW,2026-04-26 20:07:47+00:00,2026-04-26 20:19:02+00:00,False,NaT,NaT,SWR5BW_2837,2026-04-26 18:32:57+00:00,2026-04-26 20:19:05+00:00,5690.0
1241,1241,SWR1DT_3262,SWR1DT,2026-04-25 11:14:38+00:00,2026-04-25 11:32:38+00:00,True,2026-04-25 11:14:38+00:00,2026-04-25 11:15:25+00:00,SWR1DT_3304,2026-04-25 09:43:01+00:00,2026-04-25 11:32:38+00:00,5497.0
53,53,AFR78RM_223,AFR78RM,2026-04-23 09:29:35+00:00,2026-04-23 09:45:05+00:00,True,2026-04-23 09:31:20+00:00,2026-04-23 09:32:36+00:00,AFR78RM_224,2026-04-23 07:58:54+00:00,2026-04-23 09:45:05+00:00,5441.0
1879,1879,SWR5BW_2772,SWR5BW,2026-04-25 20:00:34+00:00,2026-04-25 20:09:01+00:00,False,NaT,NaT,SWR5BW_2781,2026-04-25 18:31:12+00:00,2026-04-25 20:09:02+00:00,5362.0
2470,2470,VJH365_596,VJH365,2026-04-24 08:00:20+00:00,2026-04-24 08:29:31+00:00,False,NaT,NaT,VJH365_606,2026-04-24 06:32:13+00:00,2026-04-24 08:29:31+00:00,5287.0
904,904,N169QS_6024,N169QS,2026-04-20 13:51:41+00:00,2026-04-20 14:03:25+00:00,False,NaT,NaT,N169QS_6118,2026-04-20 12:25:16+00:00,2026-04-20 14:03:26+00:00,5185.0
1836,1836,SWR54B_1938,SWR54B,2026-04-24 11:31:13+00:00,2026-04-24 11:46:14+00:00,True,2026-04-24 11:31:13+00:00,2026-04-24 11:32:45+00:00,SWR54B_1976,2026-04-24 10:07:21+00:00,2026-04-24 11:46:14+00:00,5032.0


## 8. Build Matched History Dataset

Keep only history trajectories that were successfully matched to a reviewed raw-data flight.

In [40]:
matched_flight_ids = (
    mapping_set["Flight ID New"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

if not matched_flight_ids:
    raise RuntimeError(
        "No reviewed flights could be matched to the history data."
    )

final_trajs = trajs[matched_flight_ids]

print(
    f"Matched history dataset contains "
    f"{final_trajs.data['flight_id'].nunique():,} flights"
)

final_trajs


Matched history dataset contains 2,490 flights


,count
flight_id,
MSR774_008,7746
SWR4ZW_3320,7447
VJH365_606,7039
SWR9GE_3340,6878
ULC7P_5301,6857
SWR1DT_3304,6578
SWR8YE_2886,6455
SWR594C_2706,6382
AFR78RM_224,6372


## 9. Trim History Trajectories

Some history trajectories begin before the corresponding raw-data trajectory.  
For these flights, remove history points before the raw-data start time so that no points outside the reviewed labeling interval can be mislabeled.

In [41]:
trim_map = (
    mapping_set[
        (mapping_set["Time Difference"] > 0)
        & mapping_set["Flight ID New"].notna()
    ]
    .drop_duplicates(subset="Flight ID New")
    .set_index("Flight ID New")["Start"]
    .to_dict()
)

print(f"Flights requiring start trimming: {len(trim_map):,}")


Flights requiring start trimming: 778


In [42]:
def trim_if_needed(flight):
    flight_id = flight.flight_id

    if flight_id not in trim_map:
        return flight

    start_time = trim_map[flight_id] - pd.Timedelta("1s")

    return flight.between(
        start_time,
        flight.stop,
    )


final_trajs = (
    final_trajs
    .pipe(trim_if_needed)
    .eval(
        max_workers=max_workers,
        desc="Trimming history trajectories",
    )
)

final_trajs


Trimming history trajectories:   0%|          | 0/2490 [00:00<?, ?it/s]

,count
flight_id,
SWR5MT_3402,4982
SWR736_2841,4096
THY7XK_5010,4017
SWR1DT_3245,3691
CSW2KP_4291,3608
SWR3LA_3178,3587
SXS3SB_5131,3576
SWR8YE_2942,3379
ULC7P_5304,3211


## 10. Add Pointwise Pushback Labels

Transfer the reviewed pushback start and stop timestamps from the raw-data mapping to every matched history trajectory.

In [43]:
mapping_tmp = (
    mapping_set[
        mapping_set["Flight ID New"].notna()
    ]
    .drop_duplicates(subset="Flight ID New")
    .set_index("Flight ID New")
)

pushback_start = final_trajs.data["flight_id"].map(
    mapping_tmp["Pushback Start"]
)
pushback_stop = final_trajs.data["flight_id"].map(
    mapping_tmp["Pushback Stop"]
)

final_trajs.data["Pushback"] = (
    (final_trajs.data["timestamp"] >= pushback_start)
    & (final_trajs.data["timestamp"] <= pushback_stop)
).fillna(False)

print(
    f"Pushback points: "
    f"{int(final_trajs.data['Pushback'].sum()):,}"
)


Pushback points: 132,188


## 11. Add Dataset Metadata

In [44]:
final_trajs.data["Airport"] = airport_str
final_trajs.data["Season"] = season

final_trajs.data[
    [
        "flight_id",
        "timestamp",
        "Pushback",
        "Airport",
        "Season",
    ]
].head()


,flight_id,timestamp,Pushback,Airport,Season
0,9AJIP_5415,2026-04-20 11:38:47+00:00,False,LSZH,Spring
1,9AJIP_5415,2026-04-20 11:38:48+00:00,False,LSZH,Spring
2,9AJIP_5415,2026-04-20 11:38:49+00:00,False,LSZH,Spring
3,9AJIP_5415,2026-04-20 11:38:50+00:00,False,LSZH,Spring
4,9AJIP_5415,2026-04-20 11:38:51+00:00,False,LSZH,Spring


## 12. Save Matched Dataset

Store the matched and pointwise-labeled history trajectories in `04_matched`.

In [45]:
final_trajs.to_parquet(matched_path)

print(f"Saved matched dataset: {matched_path}")
print(
    f"Flights: {final_trajs.data['flight_id'].nunique():,} | "
    f"Points: {len(final_trajs.data):,}"
)


Saved matched dataset: /cluster/home/hanv/Projects/OSN_pushback_detection/data/train_validation/04_matched/LSZH/Spring/LSZH_Spring_matched.parquet
Flights: 2,490 | Points: 2,656,336
